# ML Zoomcamp 2026 — Homework 6: Trees
Predict `fuel_efficiency_mpg` with decision trees, random forests and XGBoost.

In [ ]:
# !pip install xgboost
# !wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

## Preparing the dataset

In [ ]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df = df.fillna(0)
print(df.shape)

(10000, 11)


In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

target = 'fuel_efficiency_mpg'
y_train = df_train[target].values
y_val = df_val[target].values
df_train = df_train.drop(columns=[target])
df_val = df_val.drop(columns=[target])

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
features = list(dv.get_feature_names_out())

## Question 1 — decision tree with `max_depth=1`

In [ ]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=features))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]


**Answer:** `model_year`

## Question 2 — random forest, 10 trees

In [ ]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
print(root_mean_squared_error(y_val, rf.predict(X_val)))

1.8332197358745623


**Answer:** 1.837 (closest option)

## Question 3 — `n_estimators`

In [ ]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    score = root_mean_squared_error(y_val, rf.predict(X_val))
    print(n, round(score, 3))

10 1.833
50 1.774
100 1.77
150 1.769


**Answer:** 150

## Question 4 — best `max_depth` by mean RMSE

In [ ]:
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        scores.append(root_mean_squared_error(y_val, rf.predict(X_val)))
    print(d, np.mean(scores))

10 1.7543924943801008
15 1.7842243445385957
20 1.7867711878395158
25 1.7863054936840876


**Answer:** 10

## Question 5 — feature importance

In [ ]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances.head(8))
print(importances[['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']])

model_year                      0.296107
vehicle_weight                  0.203313
fuel_type=Gasoline              0.160712
horsepower                      0.078271
engine_displacement             0.061281
acceleration                    0.054998
origin=USA                      0.042152
drivetrain=Front-wheel drive    0.020062


**Answer:** `vehicle_weight` (most important of the four options)

## Question 6 — XGBoost `eta`
*Run these cells yourself: compare the last validation RMSE for each eta.*

In [ ]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

def train_xgb(eta):
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, verbose_eval=False)
    return root_mean_squared_error(y_val, model.predict(dval))

for eta in [0.3, 0.1]:
    print(eta, round(train_xgb(eta), 4))